In [5]:
# ======================================================================
# Q-FLOODFUSION
# Quantum-Enhanced Multimodal Flood Segmentation
#
# SEN1FLOODS11 8-CHANNEL DATASET
#
# FINAL SINGLE-CELL KAGGLE IMPLEMENTATION
#
# Dataset:
#   8 channels:
#       1. Sentinel-1 VV
#       2. Sentinel-1 VH
#       3. Sentinel-2 Green
#       4. Sentinel-2 Red
#       5. Sentinel-2 NIR
#       6. Sentinel-2 SWIR
#       7. DEM
#       8. 7-day cumulative precipitation
#
# Architecture:
#
#   8-channel input
#          |
#       CNN Encoder
#          |
#      256-D bottleneck
#          |
#      Global pooling
#          |
#        4 values
#          |
#     4-Qubit VQC
#          |
#      4 quantum features
#          |
#     Quantum expansion
#          |
#     CNN + Quantum fusion
#          |
#       U-Net decoder
#          |
#   Flood segmentation
#
# IMPORTANT:
#   CNN       -> Tesla T4 / CUDA
#   Quantum   -> CPU default.qubit
#
# The quantum circuit is executed sample-by-sample to avoid
# PennyLane batch-indexing/device compatibility issues.
# ======================================================================


# ======================================================================
# 0. INSTALL DEPENDENCIES
# ======================================================================

import os
import sys
import glob
import random
import warnings
import subprocess
import time
import math

warnings.filterwarnings("ignore")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pennylane",
        "rasterio"
    ],
    check=False
)


# ======================================================================
# 1. IMPORTS
# ======================================================================

import numpy as np
import pandas as pd
import rasterio

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import pennylane as qml

from sklearn.model_selection import train_test_split


# ======================================================================
# 2. REPRODUCIBILITY
# ======================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True


# ======================================================================
# 3. DEVICE CONFIGURATION
# ======================================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 72)
print("Q-FLOODFUSION")
print("Quantum-Enhanced Multimodal Flood Segmentation")
print("=" * 72)

print()
print("PyTorch version :", torch.__version__)
print("PennyLane       :", qml.__version__)
print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "GPU             :",
        torch.cuda.get_device_name(0)
    )

    print(
        "VRAM            :",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

print(
    "Classical device:",
    DEVICE
)


# ======================================================================
# 4. DATASET PATHS
# ======================================================================

DATA_ROOT = (
    "/kaggle/input/datasets/"
    "oindrieelmondal/"
    "sen1floods11-8-channel-remote-sensing-dataset/"
    "dataset/"
    "Sen1Floods11_8Channel"
)

IMAGE_DIR = os.path.join(
    DATA_ROOT,
    "image"
)

LABEL_DIR = os.path.join(
    DATA_ROOT,
    "label"
)

print()
print("=" * 72)
print("DATASET")
print("=" * 72)

print(
    "Image directory :",
    IMAGE_DIR
)

print(
    "Label directory :",
    LABEL_DIR
)

if not os.path.isdir(IMAGE_DIR):

    raise FileNotFoundError(
        f"Image directory not found:\n{IMAGE_DIR}"
    )

if not os.path.isdir(LABEL_DIR):

    raise FileNotFoundError(
        f"Label directory not found:\n{LABEL_DIR}"
    )


# ======================================================================
# 5. FIND GEOTIFF FILES
# ======================================================================

image_files = sorted(
    glob.glob(
        os.path.join(
            IMAGE_DIR,
            "*.tif"
        )
    )
)

label_files = sorted(
    glob.glob(
        os.path.join(
            LABEL_DIR,
            "*.tif"
        )
    )
)

print()
print(
    "Images found :",
    len(image_files)
)

print(
    "Labels found :",
    len(label_files)
)


# ======================================================================
# 6. IMAGE/LABEL ID EXTRACTION
# ======================================================================

def extract_id(path):

    filename = os.path.basename(path)

    return filename.split("_")[0]


image_dict = {
    extract_id(path): path
    for path in image_files
}

label_dict = {
    extract_id(path): path
    for path in label_files
}


common_ids = sorted(
    set(image_dict.keys())
    &
    set(label_dict.keys())
)


pairs = [
    (
        image_dict[idx],
        label_dict[idx]
    )
    for idx in common_ids
]


print(
    "Matched image-label pairs:",
    len(pairs)
)


if len(pairs) == 0:

    raise RuntimeError(
        "No image-label pairs found."
    )


# ======================================================================
# 7. VERIFY DATASET
# ======================================================================

print()
print("=" * 72)
print("DATASET VERIFICATION")
print("=" * 72)

sample_image_path, sample_label_path = pairs[0]


# ----------------------------------------------------------------------
# IMAGE
# ----------------------------------------------------------------------

with rasterio.open(
    sample_image_path
) as src:

    sample_image = src.read()

    image_shape = sample_image.shape

    # IMPORTANT:
    # NumPy ndarray has .dtype, not .dtypes
    image_dtype = sample_image.dtype

    image_crs = src.crs


# ----------------------------------------------------------------------
# LABEL
# ----------------------------------------------------------------------

with rasterio.open(
    sample_label_path
) as src:

    sample_label = src.read()

    label_shape = sample_label.shape

    # IMPORTANT:
    # NumPy ndarray has .dtype, not .dtypes
    label_dtype = sample_label.dtype


print()
print(
    "Image :",
    os.path.basename(sample_image_path)
)

print(
    "Shape :",
    image_shape
)

print(
    "Dtype :",
    image_dtype
)

print(
    "CRS   :",
    image_crs
)

print()

print(
    "Label :",
    os.path.basename(sample_label_path)
)

print(
    "Shape :",
    label_shape
)

print(
    "Dtype :",
    label_dtype
)


# ----------------------------------------------------------------------
# VALIDATION
# ----------------------------------------------------------------------

if image_shape[0] != 8:

    raise RuntimeError(
        f"Expected 8 channels but found {image_shape[0]}"
    )


if image_shape[1:] != label_shape[1:]:

    raise RuntimeError(
        "Image and label dimensions do not match."
    )


print()
print("✓ 8-channel GeoTIFF confirmed")
print("✓ Image-label pairing confirmed")


# ======================================================================
# 8. DATASET CLASS
# ======================================================================

class Sen1FloodsDataset(Dataset):

    def __init__(
        self,
        pairs,
        augment=False
    ):

        self.pairs = pairs

        self.augment = augment


    def __len__(self):

        return len(self.pairs)


    # ------------------------------------------------------------------
    # ROBUST PER-BAND NORMALIZATION
    # ------------------------------------------------------------------

    def robust_normalize(
        self,
        image
    ):

        image = image.astype(
            np.float32
        )

        output = np.zeros_like(
            image,
            dtype=np.float32
        )

        for c in range(
            image.shape[0]
        ):

            band = image[c]

            finite = np.isfinite(
                band
            )

            if not finite.any():

                output[c] = 0.0

                continue

            values = band[finite]

            p1 = np.percentile(
                values,
                1
            )

            p99 = np.percentile(
                values,
                99
            )

            if p99 <= p1:

                normalized = np.zeros_like(
                    band
                )

            else:

                normalized = (
                    band - p1
                ) / (
                    p99 - p1 + 1e-8
                )

            normalized = np.nan_to_num(
                normalized,
                nan=0.0,
                posinf=1.0,
                neginf=0.0
            )

            normalized = np.clip(
                normalized,
                0.0,
                1.0
            )

            output[c] = normalized

        return output


    # ------------------------------------------------------------------
    # GET SAMPLE
    # ------------------------------------------------------------------

    def __getitem__(
        self,
        idx
    ):

        image_path, label_path = self.pairs[idx]


        # --------------------------------------------------------------
        # READ IMAGE
        # --------------------------------------------------------------

        with rasterio.open(
            image_path
        ) as src:

            image = src.read()


        # --------------------------------------------------------------
        # READ LABEL
        # --------------------------------------------------------------

        with rasterio.open(
            label_path
        ) as src:

            label = src.read(1)


        # --------------------------------------------------------------
        # NORMALIZE IMAGE
        # --------------------------------------------------------------

        image = self.robust_normalize(
            image
        )


        # --------------------------------------------------------------
        # BINARY FLOOD LABEL
        # --------------------------------------------------------------

        label = label.astype(
            np.float32
        )

        label = (
            label > 0
        ).astype(
            np.float32
        )


        # --------------------------------------------------------------
        # AUGMENTATION
        # --------------------------------------------------------------

        if self.augment:

            # Horizontal flip
            if random.random() < 0.5:

                image = np.flip(
                    image,
                    axis=2
                ).copy()

                label = np.flip(
                    label,
                    axis=1
                ).copy()


            # Vertical flip
            if random.random() < 0.5:

                image = np.flip(
                    image,
                    axis=1
                ).copy()

                label = np.flip(
                    label,
                    axis=0
                ).copy()


            # Rotation
            if random.random() < 0.5:

                k = random.randint(
                    1,
                    3
                )

                image = np.rot90(
                    image,
                    k=k,
                    axes=(1, 2)
                ).copy()

                label = np.rot90(
                    label,
                    k=k,
                    axes=(0, 1)
                ).copy()


        # --------------------------------------------------------------
        # CONVERT TO TORCH
        # --------------------------------------------------------------

        image = torch.from_numpy(
            image.copy()
        ).float()


        label = torch.from_numpy(
            label.copy()
        ).float().unsqueeze(0)


        return image, label


# ======================================================================
# 9. TRAIN / VALIDATION / TEST SPLIT
# ======================================================================

train_pairs, temp_pairs = train_test_split(
    pairs,
    test_size=0.30,
    random_state=SEED
)

val_pairs, test_pairs = train_test_split(
    temp_pairs,
    test_size=0.50,
    random_state=SEED
)


print()
print("=" * 72)
print("DATASET SPLIT")
print("=" * 72)

print(
    "Total      :",
    len(pairs)
)

print(
    "Train      :",
    len(train_pairs)
)

print(
    "Validation :",
    len(val_pairs)
)

print(
    "Test       :",
    len(test_pairs)
)


# ======================================================================
# 10. DATASETS
# ======================================================================

train_dataset = Sen1FloodsDataset(
    train_pairs,
    augment=True
)

val_dataset = Sen1FloodsDataset(
    val_pairs,
    augment=False
)

test_dataset = Sen1FloodsDataset(
    test_pairs,
    augment=False
)


# ======================================================================
# 11. DATALOADERS
# ======================================================================

# Quantum simulator runs on CPU.
# Smaller batch prevents excessive CPU quantum simulation overhead.

BATCH_SIZE = 4

NUM_WORKERS = 2


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    drop_last=True
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)


# ======================================================================
# 12. DATA VERIFICATION
# ======================================================================

sample_x, sample_y = train_dataset[0]

print()
print("=" * 72)
print("SAMPLE VERIFICATION")
print("=" * 72)

print(
    "Input shape  :",
    tuple(sample_x.shape)
)

print(
    "Mask shape   :",
    tuple(sample_y.shape)
)

print(
    "Input range  :",
    f"{sample_x.min().item():.4f}",
    "→",
    f"{sample_x.max().item():.4f}"
)

print(
    "Flood pixels :",
    int(sample_y.sum().item())
)


batch_x, batch_y = next(
    iter(train_loader)
)


print()
print("=" * 72)
print("BATCH VERIFICATION")
print("=" * 72)

print(
    "Images :",
    tuple(batch_x.shape)
)

print(
    "Masks  :",
    tuple(batch_y.shape)
)

print(
    "GPU    :",
    torch.cuda.is_available()
)


# ======================================================================
# 13. CNN DOUBLE-CONV BLOCK
# ======================================================================

class DoubleConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.GELU(),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.GELU()
        )


    def forward(
        self,
        x
    ):

        return self.block(x)


# ======================================================================
# 14. QUANTUM CONFIGURATION
# ======================================================================

N_QUBITS = 4

Q_LAYERS = 2


# ----------------------------------------------------------------------
# QUANTUM DEVICE
#
# default.qubit is CPU based.
# ----------------------------------------------------------------------

Q_DEVICE = qml.device(
    "default.qubit",
    wires=N_QUBITS
)


# ======================================================================
# 15. SINGLE-SAMPLE QUANTUM CIRCUIT
# ======================================================================

@qml.qnode(
    Q_DEVICE,
    interface="torch",
    diff_method="backprop"
)
def quantum_circuit(
    inputs,
    weights
):

    # --------------------------------------------------------------
    # inputs shape = [4]
    #
    # IMPORTANT:
    # This QNode receives ONE sample at a time.
    # Batch processing is performed explicitly in the model.
    # --------------------------------------------------------------

    for i in range(
        N_QUBITS
    ):

        qml.RY(
            inputs[i],
            wires=i
        )

        qml.RZ(
            inputs[i],
            wires=i
        )


    # --------------------------------------------------------------
    # VARIATIONAL LAYERS
    # --------------------------------------------------------------

    for layer in range(
        Q_LAYERS
    ):

        for i in range(
            N_QUBITS
        ):

            qml.RX(
                weights[
                    layer,
                    i,
                    0
                ],
                wires=i
            )

            qml.RY(
                weights[
                    layer,
                    i,
                    1
                ],
                wires=i
            )

            qml.RZ(
                weights[
                    layer,
                    i,
                    2
                ],
                wires=i
            )


        # ----------------------------------------------------------
        # RING ENTANGLEMENT
        # ----------------------------------------------------------

        for i in range(
            N_QUBITS
        ):

            qml.CNOT(
                wires=[
                    i,
                    (i + 1) % N_QUBITS
                ]
            )


    # --------------------------------------------------------------
    # MEASUREMENTS
    # --------------------------------------------------------------

    return [
        qml.expval(
            qml.PauliZ(i)
        )
        for i in range(
            N_QUBITS
        )
    ]


# ======================================================================
# 16. PENNYLANE TORCH LAYER
# ======================================================================

weight_shapes = {
    "weights": (
        Q_LAYERS,
        N_QUBITS,
        3
    )
}


q_layer = qml.qnn.TorchLayer(
    quantum_circuit,
    weight_shapes
)


# Force quantum parameters onto CPU
q_layer = q_layer.to(
    "cpu"
)


print()
print("=" * 72)
print("QUANTUM CIRCUIT")
print("=" * 72)

print(
    "Qubits            :",
    N_QUBITS
)

print(
    "Quantum layers    :",
    Q_LAYERS
)

print(
    "Quantum simulator : default.qubit"
)

print(
    "Quantum device    : CPU"
)

print(
    "Classical device  :",
    DEVICE
)


# ======================================================================
# 17. Q-FLOODFUSION MODEL
# ======================================================================

class QFloodFusion(nn.Module):

    def __init__(self):

        super().__init__()


        # ==========================================================
        # ENCODER
        # ==========================================================

        self.enc1 = DoubleConv(
            8,
            32
        )

        self.enc2 = DoubleConv(
            32,
            64
        )

        self.enc3 = DoubleConv(
            64,
            128
        )

        self.enc4 = DoubleConv(
            128,
            256
        )


        self.pool = nn.MaxPool2d(
            2
        )


        # ==========================================================
        # QUANTUM PROJECTION
        # ==========================================================

        self.q_projection = nn.Sequential(

            nn.AdaptiveAvgPool2d(
                1
            ),

            nn.Flatten(),

            nn.Linear(
                256,
                128
            ),

            nn.GELU(),

            nn.Linear(
                128,
                N_QUBITS
            ),

            nn.Tanh()
        )


        # ==========================================================
        # QUANTUM LAYER
        # ==========================================================

        self.q_layer = q_layer


        # ==========================================================
        # QUANTUM FEATURE EXPANSION
        # ==========================================================

        self.q_expansion = nn.Sequential(

            nn.Linear(
                N_QUBITS,
                64
            ),

            nn.GELU(),

            nn.Linear(
                64,
                256
            ),

            nn.Sigmoid()
        )


        # ==========================================================
        # QUANTUM-CLASSICAL FUSION
        # ==========================================================

        self.fusion = DoubleConv(
            512,
            256
        )


        # ==========================================================
        # DECODER
        # ==========================================================

        self.up3 = nn.ConvTranspose2d(
            256,
            128,
            kernel_size=2,
            stride=2
        )

        self.dec3 = DoubleConv(
            256,
            128
        )


        self.up2 = nn.ConvTranspose2d(
            128,
            64,
            kernel_size=2,
            stride=2
        )

        self.dec2 = DoubleConv(
            128,
            64
        )


        self.up1 = nn.ConvTranspose2d(
            64,
            32,
            kernel_size=2,
            stride=2
        )

        self.dec1 = DoubleConv(
            64,
            32
        )


        # ==========================================================
        # OUTPUT
        # ==========================================================

        self.final = nn.Conv2d(
            32,
            1,
            kernel_size=1
        )


    # ==================================================================
    # FORWARD
    # ==================================================================

    def forward(
        self,
        x
    ):

        # ==========================================================
        # ENCODER
        # ==========================================================

        e1 = self.enc1(
            x
        )

        p1 = self.pool(
            e1
        )


        e2 = self.enc2(
            p1
        )

        p2 = self.pool(
            e2
        )


        e3 = self.enc3(
            p2
        )

        p3 = self.pool(
            e3
        )


        bottleneck = self.enc4(
            p3
        )


        # ==========================================================
        # CLASSICAL -> QUANTUM PROJECTION
        # ==========================================================

        q_input = self.q_projection(
            bottleneck
        )


        # [-1,1] -> [-pi,pi]
        q_input = (
            q_input *
            math.pi
        )


        # ==========================================================
        # MOVE QUANTUM INPUT TO CPU
        # ==========================================================

        q_input_cpu = q_input.float().to(
            "cpu"
        )


        # ==========================================================
        # BATCH-SAFE QUANTUM EXECUTION
        #
        # IMPORTANT:
        #
        # PennyLane QNode above expects:
        #
        #       [4]
        #
        # NOT:
        #
        #       [batch,4]
        #
        # Therefore we explicitly execute one sample at a time.
        # ==========================================================

        q_outputs = []


        for sample in q_input_cpu:

            q_out = self.q_layer(
                sample
            )

            q_outputs.append(
                q_out
            )


        # ==========================================================
        # STACK QUANTUM OUTPUTS
        # ==========================================================

        q_features_cpu = torch.stack(
            q_outputs,
            dim=0
        )


        # ==========================================================
        # CPU -> GPU
        # ==========================================================

        q_features = q_features_cpu.float().to(
            bottleneck.device
        )


        # ==========================================================
        # QUANTUM FEATURE EXPANSION
        # ==========================================================

        q_gate = self.q_expansion(
            q_features
        )


        # ==========================================================
        # BROADCAST QUANTUM REPRESENTATION
        # ==========================================================

        q_gate = q_gate.unsqueeze(
            -1
        ).unsqueeze(
            -1
        )


        q_gate = q_gate.expand(
            -1,
            -1,
            bottleneck.shape[2],
            bottleneck.shape[3]
        )


        # ==========================================================
        # FUSION
        # ==========================================================

        fused = torch.cat(
            [
                bottleneck,
                q_gate
            ],
            dim=1
        )


        fused = self.fusion(
            fused
        )


        # ==========================================================
        # DECODER 3
        # ==========================================================

        d3 = self.up3(
            fused
        )

        d3 = F.interpolate(
            d3,
            size=e3.shape[2:],
            mode="bilinear",
            align_corners=False
        )

        d3 = torch.cat(
            [
                d3,
                e3
            ],
            dim=1
        )

        d3 = self.dec3(
            d3
        )


        # ==========================================================
        # DECODER 2
        # ==========================================================

        d2 = self.up2(
            d3
        )

        d2 = F.interpolate(
            d2,
            size=e2.shape[2:],
            mode="bilinear",
            align_corners=False
        )

        d2 = torch.cat(
            [
                d2,
                e2
            ],
            dim=1
        )

        d2 = self.dec2(
            d2
        )


        # ==========================================================
        # DECODER 1
        # ==========================================================

        d1 = self.up1(
            d2
        )

        d1 = F.interpolate(
            d1,
            size=e1.shape[2:],
            mode="bilinear",
            align_corners=False
        )

        d1 = torch.cat(
            [
                d1,
                e1
            ],
            dim=1
        )

        d1 = self.dec1(
            d1
        )


        # ==========================================================
        # OUTPUT
        # ==========================================================

        logits = self.final(
            d1
        )


        # Exact input spatial resolution
        logits = F.interpolate(
            logits,
            size=x.shape[2:],
            mode="bilinear",
            align_corners=False
        )


        return logits


# ======================================================================
# 18. CREATE MODEL
# ======================================================================

model = QFloodFusion()


# Move classical model to GPU
model = model.to(
    DEVICE
)


# IMPORTANT:
# model.to(DEVICE) moves all modules including q_layer.
# Move quantum layer BACK to CPU.

model.q_layer = model.q_layer.to(
    "cpu"
)


print()
print("=" * 72)
print("MODEL")
print("=" * 72)


total_params = sum(
    p.numel()
    for p in model.parameters()
)


trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    "Total parameters     :",
    f"{total_params:,}"
)

print(
    "Trainable parameters :",
    f"{trainable_params:,}"
)


# ======================================================================
# 19. VERIFY QUANTUM PARAMETER DEVICE
# ======================================================================

quantum_devices = sorted(
    set(
        str(p.device)
        for p in model.q_layer.parameters()
    )
)


print(
    "Quantum parameter device:",
    quantum_devices
)


# ======================================================================
# 20. FORWARD PASS TEST
# ======================================================================

print()
print("=" * 72)
print("FORWARD PASS TEST")
print("=" * 72)


# Use only TWO samples for the initial test
# to minimize quantum simulation time.

test_input = batch_x[:2].to(
    DEVICE,
    non_blocking=True
)


model.eval()


with torch.no_grad():

    start = time.time()

    test_output = model(
        test_input
    )

    forward_time = (
        time.time()
        -
        start
    )


print()
print(
    "Input  shape :",
    tuple(test_input.shape)
)

print(
    "Output shape :",
    tuple(test_output.shape)
)

print(
    "Forward time :",
    f"{forward_time:.3f} sec"
)


expected_shape = (
    test_input.shape[0],
    1,
    test_input.shape[2],
    test_input.shape[3]
)


if test_output.shape != expected_shape:

    raise RuntimeError(
        f"Output shape mismatch!\n"
        f"Expected: {expected_shape}\n"
        f"Got     : {tuple(test_output.shape)}"
    )


print()
print("✓ Forward pass successful")
print("✓ CUDA/CPU quantum boundary successful")
print("✓ Batch-safe quantum execution successful")


# ======================================================================
# 21. LOSS FUNCTIONS
# ======================================================================

def dice_loss(
    logits,
    targets,
    smooth=1.0
):

    probs = torch.sigmoid(
        logits
    )


    probs = probs.contiguous().view(
        probs.shape[0],
        -1
    )


    targets = targets.contiguous().view(
        targets.shape[0],
        -1
    )


    intersection = (
        probs * targets
    ).sum(
        dim=1
    )


    dice = (
        2.0 * intersection + smooth
    ) / (
        probs.sum(dim=1)
        +
        targets.sum(dim=1)
        +
        smooth
    )


    return (
        1.0 -
        dice.mean()
    )


def segmentation_loss(
    logits,
    targets
):

    bce = F.binary_cross_entropy_with_logits(
        logits,
        targets
    )


    dice = dice_loss(
        logits,
        targets
    )


    return (
        0.5 * bce
        +
        0.5 * dice
    )


# ======================================================================
# 22. METRICS
# ======================================================================

@torch.no_grad()
def calculate_metrics(
    logits,
    targets,
    threshold=0.5
):

    probabilities = torch.sigmoid(
        logits
    )


    predictions = (
        probabilities >= threshold
    ).float()


    targets = (
        targets >= 0.5
    ).float()


    predictions = predictions.view(
        -1
    )

    targets = targets.view(
        -1
    )


    tp = (
        (predictions == 1)
        &
        (targets == 1)
    ).sum().item()


    tn = (
        (predictions == 0)
        &
        (targets == 0)
    ).sum().item()


    fp = (
        (predictions == 1)
        &
        (targets == 0)
    ).sum().item()


    fn = (
        (predictions == 0)
        &
        (targets == 1)
    ).sum().item()


    eps = 1e-8


    precision = (
        tp /
        (
            tp + fp + eps
        )
    )


    recall = (
        tp /
        (
            tp + fn + eps
        )
    )


    f1 = (
        2.0 *
        precision *
        recall
        /
        (
            precision
            +
            recall
            +
            eps
        )
    )


    iou = (
        tp /
        (
            tp
            +
            fp
            +
            fn
            +
            eps
        )
    )


    dice = (
        2.0 * tp /
        (
            2.0 * tp
            +
            fp
            +
            fn
            +
            eps
        )
    )


    accuracy = (
        tp + tn
    ) / (
        tp
        +
        tn
        +
        fp
        +
        fn
        +
        eps
    )


    return {

        "accuracy":
            accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "iou":
            iou,

        "dice":
            dice
    }


# ======================================================================
# 23. OPTIMIZER
# ======================================================================

LEARNING_RATE = 1e-3

WEIGHT_DECAY = 1e-4


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


EPOCHS = 20


scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)


# ======================================================================
# 24. AMP
# ======================================================================

if torch.cuda.is_available():

    scaler = torch.amp.GradScaler(
        "cuda"
    )

else:

    scaler = None


# ======================================================================
# 25. TRAINING FUNCTION
# ======================================================================

def train_one_epoch(
    model,
    loader,
    optimizer
):

    model.train()


    running_loss = 0.0


    metric_sum = {

        "accuracy": 0.0,

        "precision": 0.0,

        "recall": 0.0,

        "f1": 0.0,

        "iou": 0.0,

        "dice": 0.0
    }


    batches = 0


    for images, masks in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )


        masks = masks.to(
            DEVICE,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        # ----------------------------------------------------------
        # CLASSICAL CNN AMP
        # ----------------------------------------------------------

        if torch.cuda.is_available():

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                logits = model(
                    images
                )

                loss = segmentation_loss(
                    logits,
                    masks
                )

        else:

            logits = model(
                images
            )

            loss = segmentation_loss(
                logits,
                masks
            )


        # ----------------------------------------------------------
        # BACKPROPAGATION
        # ----------------------------------------------------------

        if scaler is not None:

            scaler.scale(
                loss
            ).backward()


            scaler.unscale_(
                optimizer
            )


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )


            scaler.step(
                optimizer
            )


            scaler.update()

        else:

            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )


            optimizer.step()


        # ----------------------------------------------------------
        # METRICS
        # ----------------------------------------------------------

        metrics = calculate_metrics(
            logits.float(),
            masks
        )


        running_loss += (
            loss.item()
        )


        for key in metric_sum:

            metric_sum[key] += (
                metrics[key]
            )


        batches += 1


    results = {

        "loss":
            running_loss / batches
    }


    for key in metric_sum:

        results[key] = (
            metric_sum[key]
            /
            batches
        )


    return results


# ======================================================================
# 26. VALIDATION FUNCTION
# ======================================================================

@torch.no_grad()
def validate(
    model,
    loader
):

    model.eval()


    running_loss = 0.0


    metric_sum = {

        "accuracy": 0.0,

        "precision": 0.0,

        "recall": 0.0,

        "f1": 0.0,

        "iou": 0.0,

        "dice": 0.0
    }


    batches = 0


    for images, masks in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )


        masks = masks.to(
            DEVICE,
            non_blocking=True
        )


        logits = model(
            images
        )


        loss = segmentation_loss(
            logits,
            masks
        )


        metrics = calculate_metrics(
            logits.float(),
            masks
        )


        running_loss += (
            loss.item()
        )


        for key in metric_sum:

            metric_sum[key] += (
                metrics[key]
            )


        batches += 1


    results = {

        "loss":
            running_loss / batches
    }


    for key in metric_sum:

        results[key] = (
            metric_sum[key]
            /
            batches
        )


    return results


# ======================================================================
# 27. TRAINING LOOP
# ======================================================================

print()
print("=" * 72)
print("TRAINING")
print("=" * 72)


best_val_iou = -1.0

best_state = None

history = []


for epoch in range(
    1,
    EPOCHS + 1
):

    epoch_start = time.time()


    # --------------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------------

    train_results = train_one_epoch(
        model,
        train_loader,
        optimizer
    )


    # --------------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------------

    val_results = validate(
        model,
        val_loader
    )


    # --------------------------------------------------------------
    # SCHEDULER
    # --------------------------------------------------------------

    scheduler.step()


    # --------------------------------------------------------------
    # BEST MODEL
    # --------------------------------------------------------------

    if (
        val_results["iou"]
        >
        best_val_iou
    ):

        best_val_iou = (
            val_results["iou"]
        )


        best_state = {

            k:
            v.detach().cpu().clone()

            for k, v
            in model.state_dict().items()
        }


        torch.save(
            {
                "epoch":
                    epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "val_iou":
                    best_val_iou
            },

            "/kaggle/working/"
            "qfloodfusion_best.pth"
        )


        best_marker = " ★ BEST"


    else:

        best_marker = ""


    # --------------------------------------------------------------
    # TIME
    # --------------------------------------------------------------

    elapsed = (
        time.time()
        -
        epoch_start
    )


    # --------------------------------------------------------------
    # HISTORY
    # --------------------------------------------------------------

    history.append({

        "epoch":
            epoch,

        "train_loss":
            train_results["loss"],

        "train_dice":
            train_results["dice"],

        "train_iou":
            train_results["iou"],

        "train_f1":
            train_results["f1"],

        "val_loss":
            val_results["loss"],

        "val_dice":
            val_results["dice"],

        "val_iou":
            val_results["iou"],

        "val_f1":
            val_results["f1"],

        "val_precision":
            val_results["precision"],

        "val_recall":
            val_results["recall"],

        "learning_rate":
            optimizer.param_groups[0]["lr"],

        "time_sec":
            elapsed
    })


    # --------------------------------------------------------------
    # LOG
    # --------------------------------------------------------------

    print(

        f"Epoch "
        f"{epoch:02d}/{EPOCHS} | "

        f"Train Loss "
        f"{train_results['loss']:.4f} | "

        f"Train Dice "
        f"{train_results['dice']:.4f} | "

        f"Train IoU "
        f"{train_results['iou']:.4f} | "

        f"Val Loss "
        f"{val_results['loss']:.4f} | "

        f"Val Dice "
        f"{val_results['dice']:.4f} | "

        f"Val IoU "
        f"{val_results['iou']:.4f} | "

        f"Val F1 "
        f"{val_results['f1']:.4f} | "

        f"{elapsed:.1f}s"

        f"{best_marker}"
    )


# ======================================================================
# 28. RESTORE BEST MODEL
# ======================================================================

if best_state is not None:

    model.load_state_dict(
        best_state
    )


print()
print("=" * 72)
print("BEST MODEL RESTORED")
print("=" * 72)

print(
    "Best validation IoU:",
    f"{best_val_iou:.4f}"
)


# ======================================================================
# 29. FINAL TEST
# ======================================================================

print()
print("=" * 72)
print("FINAL TEST EVALUATION")
print("=" * 72)


test_results = validate(
    model,
    test_loader
)


print()
print(
    "Test Loss      :",
    f"{test_results['loss']:.4f}"
)

print(
    "Test Accuracy  :",
    f"{test_results['accuracy']:.4f}"
)

print(
    "Test Precision :",
    f"{test_results['precision']:.4f}"
)

print(
    "Test Recall    :",
    f"{test_results['recall']:.4f}"
)

print(
    "Test F1        :",
    f"{test_results['f1']:.4f}"
)

print(
    "Test IoU       :",
    f"{test_results['iou']:.4f}"
)

print(
    "Test Dice      :",
    f"{test_results['dice']:.4f}"
)


# ======================================================================
# 30. SAVE HISTORY
# ======================================================================

history_df = pd.DataFrame(
    history
)


history_df.to_csv(
    "/kaggle/working/"
    "qfloodfusion_training_history.csv",
    index=False
)


# ======================================================================
# 31. SAVE FINAL MODEL
# ======================================================================

torch.save(
    {

        "model_state_dict":
            model.state_dict(),

        "test_results":
            test_results,

        "best_val_iou":
            best_val_iou,

        "config": {

            "input_channels":
                8,

            "qubits":
                N_QUBITS,

            "quantum_layers":
                Q_LAYERS,

            "epochs":
                EPOCHS,

            "batch_size":
                BATCH_SIZE,

            "learning_rate":
                LEARNING_RATE,

            "weight_decay":
                WEIGHT_DECAY
        }

    },

    "/kaggle/working/"
    "qfloodfusion_final.pth"
)


# ======================================================================
# 32. FINAL SUMMARY
# ======================================================================

print()
print("=" * 72)
print("Q-FLOODFUSION COMPLETE")
print("=" * 72)


print()
print("DATASET")
print(
    "  Total samples :",
    len(pairs)
)

print(
    "  Train         :",
    len(train_pairs)
)

print(
    "  Validation    :",
    len(val_pairs)
)

print(
    "  Test          :",
    len(test_pairs)
)


print()
print("ARCHITECTURE")

print(
    "  Input         : 8-channel remote sensing"
)

print(
    "  Encoder       : CNN"
)

print(
    "  Bottleneck    : 256 channels"
)

print(
    "  Quantum       : 4 qubits"
)

print(
    "  Q layers      :",
    Q_LAYERS
)

print(
    "  Fusion        : CNN + quantum features"
)

print(
    "  Decoder       : U-Net style"
)

print(
    "  Output        : Binary flood segmentation"
)


print()
print("HARDWARE")

print(
    "  Classical CNN :",
    DEVICE
)

print(
    "  Quantum       : CPU default.qubit"
)


print()
print("FINAL TEST PERFORMANCE")

print(
    "  IoU           :",
    f"{test_results['iou']:.4f}"
)

print(
    "  Dice          :",
    f"{test_results['dice']:.4f}"
)

print(
    "  F1            :",
    f"{test_results['f1']:.4f}"
)

print(
    "  Precision     :",
    f"{test_results['precision']:.4f}"
)

print(
    "  Recall        :",
    f"{test_results['recall']:.4f}"
)


print()
print("SAVED FILES")

print(
    "  qfloodfusion_best.pth"
)

print(
    "  qfloodfusion_final.pth"
)

print(
    "  qfloodfusion_training_history.csv"
)


print()
print("=" * 72)
print("✓ PIPELINE FINISHED SUCCESSFULLY")
print("=" * 72)

Q-FLOODFUSION
Quantum-Enhanced Multimodal Flood Segmentation

PyTorch version : 2.10.0+cu128
PennyLane       : 0.45.1
CUDA available  : True
GPU             : Tesla T4
VRAM            : 14.56 GB
Classical device: cuda

DATASET
Image directory : /kaggle/input/datasets/oindrieelmondal/sen1floods11-8-channel-remote-sensing-dataset/dataset/Sen1Floods11_8Channel/image
Label directory : /kaggle/input/datasets/oindrieelmondal/sen1floods11-8-channel-remote-sensing-dataset/dataset/Sen1Floods11_8Channel/label

Images found : 446
Labels found : 446
Matched image-label pairs: 446

DATASET VERIFICATION

Image : 1010394_image.tif
Shape : (8, 512, 512)
Dtype : float32
CRS   : EPSG:4326

Label : 1010394_label.tif
Shape : (1, 512, 512)
Dtype : int16

✓ 8-channel GeoTIFF confirmed
✓ Image-label pairing confirmed

DATASET SPLIT
Total      : 446
Train      : 312
Validation : 67
Test       : 67

SAMPLE VERIFICATION
Input shape  : (8, 512, 512)
Mask shape   : (1, 512, 512)
Input range  : 0.0000 → 1.0000
Flo